# Week 3 - Classical Concert RAG Pipeline

클래식 공연 예습 도메인의 원문 데이터(`assignments/daexvk/data/raw/*.txt`)를 사용해 RAG 파이프라인을 직접 구성합니다.

요구사항 반영:

- Loader -> Splitter -> Embed -> Store -> Retrieve 5단계 직접 구성
- FAISS vector store 사용
- 같은 embedding 객체를 indexing/query 양쪽에 사용
- splitter 전략 2개 비교: `recursive_character`, `token`
- 비교 쿼리 3개 이상
- 2-step RAG: retrieve -> generate
- 답변에 근거 문서 파일 정보 출력
- 테스트 질문 5개 이상: 사실조회, 종합, 비교, 모호한 질문, 공연 예절


In [1]:
from pathlib import Path
import sys

cwd = Path.cwd()
candidates = [cwd, cwd / "assignments" / "daexvk" / "week3", cwd / "week3"]
target_dir = next(path for path in candidates if (path / "rag_pipeline.py").exists())
sys.path.insert(0, str(target_dir.resolve()))
print("target_dir:", target_dir.resolve())

target_dir: /Users/nozerose/Documents/GitHub/rag-agent-study/assignments/daexvk/week3


## 1. Loader

`assignments/daexvk/data/raw/*.txt`의 Wikipedia plain text 원문 28개를 로드합니다. `sources.json`은 출처/라이선스 추적용이고, RAG ingest 대상은 raw 텍스트 파일입니다.

In [2]:
from rag_pipeline import load_raw_documents

documents = load_raw_documents()
print("loaded_documents:", len(documents))
print("first_source:", documents[0].metadata["source_file"])
print("first_chars:", documents[0].page_content[:180].replace("\n", " "))

loaded_documents: 28
first_source: 01_classical_music.txt
first_chars: Classical music is a tradition of art music in the Western world, considered to be distinct from Western folk music or popular music traditions. It is sometimes distinguished as We


## 2. Splitter 전략 2개

- `recursive_character`: 문단/줄/문장 경계를 우선하는 RecursiveCharacterTextSplitter
- `token`: 외부 tokenizer 다운로드 없이 동작하는 whitespace token window splitter

In [3]:
from rag_pipeline import split_recursive, split_token

recursive_chunks = split_recursive(documents)
token_chunks = split_token(documents)
print("recursive_character chunks:", len(recursive_chunks))
print("token chunks:", len(token_chunks))
print("recursive sample:", recursive_chunks[0].metadata)
print("token sample:", token_chunks[0].metadata)

recursive_character chunks: 1158
token chunks: 861
recursive sample: {'source_file': '01_classical_music.txt', 'source_path': '/Users/nozerose/Documents/GitHub/rag-agent-study/assignments/daexvk/data/raw/01_classical_music.txt', 'title': 'Classical Music', 'strategy': 'recursive_character', 'chunk_index': 0, 'source_id': '01_classical_music.txt#0'}
token sample: {'source_file': '01_classical_music.txt', 'source_path': '/Users/nozerose/Documents/GitHub/rag-agent-study/assignments/daexvk/data/raw/01_classical_music.txt', 'title': 'Classical Music', 'token_start': 0, 'token_end': 220, 'strategy': 'token', 'chunk_index': 0, 'source_id': '01_classical_music.txt#0'}


## 3-5. Embed, Store, Retrieve

임베딩은 HuggingFace `sentence-transformers/all-MiniLM-L6-v2`를 우선 사용하도록 구현했습니다. 현재 로컬 환경에는 `sentence_transformers`가 없어 같은 index/query 인터페이스의 `sklearn` TF-IDF fallback을 사용했습니다. 두 경우 모두 같은 embedding 객체가 store 생성과 query retrieval 양쪽에 사용됩니다.

In [4]:
from rag_pipeline import build_index

recursive_index = build_index("recursive", documents, prefer_huggingface=False)
token_index = build_index("token", documents, prefer_huggingface=False)

print("recursive strategy:", recursive_index.strategy)
print("recursive embedding:", getattr(recursive_index.embeddings, "model_name", type(recursive_index.embeddings).__name__))
print("recursive vectorstore:", type(recursive_index.vectorstore).__name__)
print("token strategy:", token_index.strategy)
print("token embedding:", getattr(token_index.embeddings, "model_name", type(token_index.embeddings).__name__))
print("token vectorstore:", type(token_index.vectorstore).__name__)

recursive strategy: recursive
recursive embedding: local:sklearn-tfidf
recursive vectorstore: FAISS
token strategy: token
token embedding: local:sklearn-tfidf
token vectorstore: FAISS


## Splitter 비교: 비교 쿼리 3개 이상

같은 원문 corpus와 같은 embedding 방식에서 splitter만 바꿔 top-k 결과를 비교합니다.

In [5]:
from rag_pipeline import compare_retrievers

comparison_queries = [
    "베토벤 5번은 몇 악장이고 첫 동기는 무엇인가?",
    "교향곡과 협주곡과 실내악은 어떻게 다른가?",
    "공연장에서 언제 박수를 쳐야 하나?",
]
comparison_rows = compare_retrievers(
    {"recursive_character": recursive_index, "token": token_index},
    comparison_queries,
    k=3,
)
for row in comparison_rows:
    print("QUERY:", row["query"])
    print("STRATEGY:", row["strategy"])
    print("TOP SOURCES:", row["top_sources"])
    print("TOP TITLES:", row["top_titles"])
    print("CHUNK LENGTHS:", row["chunk_lengths"])
    print("-" * 80)

QUERY: 베토벤 5번은 몇 악장이고 첫 동기는 무엇인가?
STRATEGY: recursive_character
TOP SOURCES: ['15_symphony_no_5_beethoven.txt#735', '15_symphony_no_5_beethoven.txt#756', '15_symphony_no_5_beethoven.txt#754']
TOP TITLES: ['Symphony No 5 Beethoven', 'Symphony No 5 Beethoven', 'Symphony No 5 Beethoven']
CHUNK LENGTHS: [748, 1091, 837]
--------------------------------------------------------------------------------
QUERY: 베토벤 5번은 몇 악장이고 첫 동기는 무엇인가?
STRATEGY: token
TOP SOURCES: ['15_symphony_no_5_beethoven.txt#559', '15_symphony_no_5_beethoven.txt#558', '15_symphony_no_5_beethoven.txt#545']
TOP TITLES: ['Symphony No 5 Beethoven', 'Symphony No 5 Beethoven', 'Symphony No 5 Beethoven']
CHUNK LENGTHS: [1339, 1424, 1402]
--------------------------------------------------------------------------------
QUERY: 교향곡과 협주곡과 실내악은 어떻게 다른가?
STRATEGY: recursive_character
TOP SOURCES: ['06_chamber_music.txt#231', '06_chamber_music.txt#186', '06_chamber_music.txt#185']
TOP TITLES: ['Chamber Music', 'Chamber Music', 'Chamber

### 비교 해석

- 두 전략 모두 같은 질의에서 같은 주제 파일을 찾지만 chunk 크기와 경계가 다릅니다.
- `recursive_character`는 문단 경계를 유지해 문맥이 자연스럽지만 chunk 수가 더 많습니다.
- `token`은 chunk 수가 더 적고 길이가 비교적 일정하지만 문장/문단 중간에서 잘릴 수 있습니다.
- 영어 원문 corpus에 한국어 질의를 던지기 때문에 fallback TF-IDF에서는 `expand_query()`로 한국어 키워드를 영어 작품명/장르명으로 확장했습니다.

## 2-step RAG: Retrieve -> Generate

아래 함수는 먼저 retrieve로 근거 chunk를 가져오고, 그 다음 generate 단계에서 답변과 근거 문서 파일 정보를 함께 출력합니다. 로컬 재현성을 위해 이 노트북 출력은 `use_llm=False` extractive generation으로 남겼습니다. API 키가 있으면 `use_llm=True`로 바꾸어 LLM 답변을 생성할 수 있습니다.

In [6]:
from rag_pipeline import rag_answer

sample_question = "말러 교향곡 5번 공연 전에 어디를 듣고 가면 좋을까?"
sample_result = rag_answer(recursive_index, sample_question, k=3, use_llm=False)
print(sample_result["answer"][:2200])

질문: 말러 교향곡 5번 공연 전에 어디를 듣고 가면 좋을까?

검색된 원문 근거에서 바로 확인되는 내용은 아래와 같습니다.
- == Structure == The symphony is generally regarded as the most conventional symphony that he had yet written, but from such an unconventional composer it still had many peculiarities. It almost has a four-movement structure, as the first two can easily be viewed as essentially a whole. The symphony also ends with a rondo, in the classical style. Some peculiarities are the funeral march that opens the piece and the Adagietto for harp and strings that contrasts with the complex orchestration of the other movements. A performance of the symphony l
- The Symphony No. 5 in C# minor by Gustav Mahler was composed in 1901 and 1902, mostly during the summer months at Mahler's holiday cottage at Maiernigg. Among its most distinctive features are the trumpet solo that opens the work with a rhythmic motif similar to the opening of Ludwig van Beethoven's Symphony No. 5, the horn solos in the third movement and the frequently pe

## 테스트 질문 5개 이상

의도적으로 유형을 나누었습니다.

1. 사실조회
2. 종합
3. 비교
4. 모호한 질문
5. 공연 예절

In [7]:
test_questions = [
    ("사실조회", "베토벤 5번은 몇 악장이고 첫 동기는 무엇인가?"),
    ("종합", "말러 교향곡 5번 공연 전에 어디를 듣고 가면 좋을까?"),
    ("비교", "교향곡, 협주곡, 실내악은 공연장에서 듣는 초점이 어떻게 달라?"),
    ("모호한 질문", "운명 그거 공연 전에 뭐만 알고 가면 돼?"),
    ("공연 예절", "클래식 공연 처음인데 언제 박수치고 복장은 어떻게 하면 돼?"),
]

for question_type, question in test_questions:
    result = rag_answer(recursive_index, question, k=3, use_llm=False)
    print("TYPE:", question_type)
    print("QUESTION:", question)
    print("SOURCES:")
    print(result["sources"])
    print("ANSWER PREVIEW:")
    print(result["answer"][:650].replace("\n\n", "\n"))
    print("=" * 80)

TYPE: 사실조회
QUESTION: 베토벤 5번은 몇 악장이고 첫 동기는 무엇인가?
SOURCES:
[1] 15_symphony_no_5_beethoven.txt chunk=735 strategy=recursive_character
[2] 15_symphony_no_5_beethoven.txt chunk=756 strategy=recursive_character
[3] 15_symphony_no_5_beethoven.txt chunk=754 strategy=recursive_character
ANSWER PREVIEW:
질문: 베토벤 5번은 몇 악장이고 첫 동기는 무엇인가?
검색된 원문 근거에서 바로 확인되는 내용은 아래와 같습니다.
- The Symphony No. 5 in C minor, Op. 67 (occasionally known as the Fate Symphony, German: Schicksalssinfonie), is a symphony composed by Ludwig van Beethoven between 1804 and 1808. It is one of the best-known of all symphonies and one of the most frequently played. First performed in Vienna in 1808, the work achieved its strong critical reputation not long afterward; E. T. A. Hoffmann described it as "one of the most important works of the time".  The 5th Symphony has 4 movements. It begins with a distinctive 4-note "short-short-short-long" motif
- === Repetition of the opening
TYPE: 종합
QUESTION: 말러 교향곡 5번 공연 전에 어디를 듣고 가면 좋을까?
SOURC

## 정리

- 원문 loader, splitter 2종, embedding, FAISS store, retrieve를 직접 구성했습니다.
- recursive/token splitter의 top source 차이를 3개 이상의 쿼리로 비교했습니다.
- 2-step RAG는 `rag_answer()` 내부에서 retrieve 후 generate를 수행합니다.
- 답변에는 근거 문서 파일명과 chunk index를 함께 출력합니다.
- 5개 테스트 질문으로 사실조회, 종합, 비교, 모호한 질문, 공연 예절 유형을 확인했습니다.